> **Deprecated**
>
> This notebook has been consolidated into `secondary_documentation_notebook.ipynb`.
> Please refer to that notebook instead.
> 
> Section: **Extracting Sample Images for Documentation** (Section 6 in `secondary_documentation_notebook.ipynb`).

In [1]:
from rasterio.warp import transform_bounds, reproject, Resampling
from rasterio.transform import from_origin
import numpy as np
import rasterio


SHAPE_OF_INTEREST = [
    {
        "type": "Polygon",
        "coordinates": [[((5, 46)), (5, 36), (18.5, 36), (18.5, 46)]],
    }
]

x_center = 0
y_center = 0

for pair in SHAPE_OF_INTEREST[0]["coordinates"][0]:
    x_center += pair[0] / 4
    y_center += pair[1] / 4


dst_crs = f"+proj=aeqd +lat_0={y_center} +lon_0={x_center} +x_0=0 +y_0=0 +datum=WGS84 +units=m +no_defs"


def build_fixed_grid(resolution_m):
    """Zielgitter nur aus dem Polygon und der Auflösung – unabhängig vom Input."""
    coords = SHAPE_OF_INTEREST[0]["coordinates"][0]
    lons = [c[0] for c in coords]
    lats = [c[1] for c in coords]

    # Polygon-Bounds (Grad) in die Ziel-CRS (Meter) umrechnen
    left, bottom, right, top = transform_bounds(
        "EPSG:4326",
        dst_crs,
        min(lons),
        min(lats),
        max(lons),
        max(lats),
        densify_pts=21,
    )

    # Auf die Auflösung einrasten -> Ursprung sitzt bei jeder Datei gleich
    left = np.floor(left / resolution_m) * resolution_m
    bottom = np.floor(bottom / resolution_m) * resolution_m
    right = np.ceil(right / resolution_m) * resolution_m
    top = np.ceil(top / resolution_m) * resolution_m

    width = int((right - left) / resolution_m)
    height = int((top - bottom) / resolution_m)
    transform = from_origin(left, top, resolution_m, resolution_m)
    return transform, width, height


def export_area(input_file: str, output_file: str, isdem, resolution_m=1000):
    transform, width, height = build_fixed_grid(resolution_m)

    with rasterio.open(input_file) as src:
        out_meta = src.meta.copy()
        out_meta.update(
            {
                "crs": dst_crs,
                "transform": transform,
                "width": width,
                "height": height,
                "count": src.count,
                "dtype": "float16",
            }
        )

        # Jede Quelle in genau dieses feste Gitter projizieren (als float32)
        # Nicht abgedeckte Pixel bleiben NaN, damit sie die Streckung nicht verfälschen
        projected = np.full((src.count, height, width), np.nan, dtype="float32")
        for i in range(src.count):
            reproject(
                source=rasterio.band(src, i + 1),
                destination=projected[i],
                src_transform=src.transform,
                src_crs=src.crs,
                dst_transform=transform,
                dst_crs=dst_crs,
                dst_nodata=np.nan,
                resampling=Resampling.bilinear if isdem else Resampling.nearest,
            )

        with rasterio.open(output_file, "w", **out_meta) as destination:
            for i in range(src.count):
                if isdem:
                    band = np.clip(projected[i], 0, None)  # Negatives -> 0 (wie gehabt)
                    vmin = np.nanmin(band)
                    vmax = np.nanmax(band)
                    scaled = band / (vmax - vmin) * 255 *255 # deine Formel
                    out = np.nan_to_num(scaled, nan=0).astype("float16")
                else:
                    out = (
                        np.nan_to_num(projected[i], nan=0).clip(0, 255).astype("uint8")
                    )

                destination.write(out, i + 1)

In [9]:


export_area('/Users/scharnagl/Documents/GitHub/geospatial-landscape-clustering-by-fft/input_geotiffs/geotiff 002.2, 035.2, 020.4, 050.3.tif', "output/export.tif", True)


In [2]:


export_area("/Users/scharnagl/Documents/GitHub/geospatial-landscape-clustering-by-fft/output/label_images/geotiff 002.2, 035.2, 020.4, 050.3.tif 277c  tlszkm 12.0  tlszpx 23  fftlvls 17  6.7x -ovrlp-pct 85 fltrrds 15 lblct 10.tif", "output/export_c_X.tif", False)

In [3]:


export_area("/Users/scharnagl/Documents/GitHub/geospatial-landscape-clustering-by-fft/output/label_images/geotiff 002.2, 035.2, 020.4, 050.3.tif b5eb  tlszkm 12.0  tlszpx 23  fftlvls 17  6.7x -ovrlp-pct 85 fltrrds 15 lblct 10.tif", "output/export_b_X.tif", False)